# Lab 24: Recommendation Systems — Vectors, Similarity, and Matrix Factorization

This lab is a full computational companion to Chapter 24. We will build recommendation systems from the ground up:

1. represent ratings as a matrix with missing entries;
2. compare users and items using cosine similarity;
3. make neighborhood-based recommendations;
4. understand low-rank latent factors;
5. train a small matrix-factorization model using gradient descent;
6. evaluate predictions;
7. visualize hidden taste spaces.

The guiding idea is: **a recommender predicts unknown entries of a user--item matrix by discovering geometric structure.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. A small movie-rating matrix

Rows are users. Columns are movies. Missing ratings are represented by `np.nan`, not by zero.

In [ ]:
users = ["Ada", "Ben", "Chloe", "Diego", "Emma", "Fatima", "Gus", "Hana"]
movies = ["Space Quest", "Robot Dreams", "Alien River", "Love in Paris", "City Romance", "Castle Magic", "Dragon War"]

R = np.array([
    [5, 5, 4, 1, 1, np.nan, 4],
    [4, 5, np.nan, 1, 2, np.nan, 5],
    [1, np.nan, 2, 5, 5, 4, 1],
    [np.nan, 2, 1, 4, 5, 5, 1],
    [5, 4, 5, np.nan, 1, 1, 5],
    [1, 1, np.nan, 5, 4, 5, np.nan],
    [4, np.nan, 5, 2, np.nan, 1, 4],
    [np.nan, 1, 1, 4, 5, 4, 1]
], dtype=float)

ratings = pd.DataFrame(R, index=users, columns=movies)
ratings

## 2. Visualize observed and missing ratings

A recommendation matrix is usually sparse. Before building a model, always inspect where the data are observed.

In [ ]:
plt.figure(figsize=(9,4))
plt.imshow(~np.isnan(R), aspect='auto')
plt.xticks(range(len(movies)), movies, rotation=45, ha='right')
plt.yticks(range(len(users)), users)
plt.title("Observed entries: bright = observed, dark = missing")
plt.colorbar(label="Observed?")
plt.tight_layout()
plt.show()

## 3. User means and centered ratings

Users have different rating habits. Some rate generously; others rate harshly. We center each user's ratings by subtracting that user's mean.

In [ ]:
user_means = np.nanmean(R, axis=1)
centered = R - user_means[:, None]

pd.DataFrame(centered, index=users, columns=movies)

## 4. Cosine similarity with missing entries

To compare two users, we use only the movies both users rated. This avoids treating missing values as zeros.

In [ ]:
def cosine_on_overlap(x, y):
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() == 0:
        return np.nan
    a, b = x[mask], y[mask]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return np.nan
    return float(a @ b / denom)

S_user = np.zeros((len(users), len(users)))
for i in range(len(users)):
    for j in range(len(users)):
        S_user[i,j] = cosine_on_overlap(centered[i], centered[j])

pd.DataFrame(S_user, index=users, columns=users)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(S_user, vmin=-1, vmax=1)
plt.xticks(range(len(users)), users, rotation=45, ha='right')
plt.yticks(range(len(users)), users)
plt.title("Centered user-user cosine similarity")
plt.colorbar(label="cosine similarity")
plt.tight_layout()
plt.show()

### Student task

Find the two most similar users. What movies do they both like? Does the similarity score make sense?

In [ ]:
pairs=[]
for i in range(len(users)):
    for j in range(i+1,len(users)):
        pairs.append((S_user[i,j], users[i], users[j]))
sorted(pairs, reverse=True)[:5]

## 5. User-based collaborative filtering

We now predict missing entries using similar users. The prediction formula is

$$
\hat r_{ui}=ar r_u+rac{\sum_v s(u,v)(r_{vi}-ar r_v)}{\sum_v |s(u,v)|}.
$$

In [ ]:
def predict_user_based(R, u, i, S, user_means, min_neighbors=1):
    num = 0.0
    den = 0.0
    for v in range(R.shape[0]):
        if v == u or np.isnan(R[v, i]):
            continue
        s = S[u, v]
        if np.isnan(s) or s <= 0:
            continue
        num += s * (R[v, i] - user_means[v])
        den += abs(s)
    if den == 0:
        return user_means[u]
    return user_means[u] + num / den

pred_user = np.full_like(R, np.nan)
for u in range(R.shape[0]):
    for i in range(R.shape[1]):
        if np.isnan(R[u,i]):
            pred_user[u,i] = predict_user_based(R,u,i,S_user,user_means)

pd.DataFrame(pred_user, index=users, columns=movies)

## 6. Item-based similarity

Item-based filtering compares columns instead of rows. Two movies are similar if the same users tend to rate them similarly.

In [ ]:
item_means = np.nanmean(R, axis=0)
item_centered = R - item_means[None, :]

S_item = np.zeros((len(movies), len(movies)))
for i in range(len(movies)):
    for j in range(len(movies)):
        S_item[i,j] = cosine_on_overlap(item_centered[:,i], item_centered[:,j])

pd.DataFrame(S_item, index=movies, columns=movies)

In [ ]:
plt.figure(figsize=(7,6))
plt.imshow(S_item, vmin=-1, vmax=1)
plt.xticks(range(len(movies)), movies, rotation=45, ha='right')
plt.yticks(range(len(movies)), movies)
plt.title("Item-item cosine similarity")
plt.colorbar(label="cosine similarity")
plt.tight_layout()
plt.show()

## 7. Low-rank worlds

Recommendation works well when preferences have hidden low-dimensional structure. We simulate a world with two hidden factors: action taste and romance/fantasy taste.

In [ ]:
rng = np.random.default_rng(7)
true_user_factors = rng.normal(size=(30, 2))
true_item_factors = np.array([
    [2.0, -0.5], [1.8, -0.2], [2.2, -0.7],
    [-0.4, 2.0], [-0.2, 1.8], [-0.6, 2.2], [1.5, 0.8]
])
true_scores = true_user_factors @ true_item_factors.T
noisy_scores = true_scores + 0.25*rng.normal(size=true_scores.shape)

plt.figure(figsize=(6,5))
plt.scatter(true_user_factors[:,0], true_user_factors[:,1])
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("Hidden factor 1")
plt.ylabel("Hidden factor 2")
plt.title("Users in a two-dimensional hidden taste space")
plt.tight_layout()
plt.show()

## 8. Matrix factorization by gradient descent

We learn user factors $P$ and item factors $Q$ from observed ratings only.

In [ ]:
def train_matrix_factorization(R, k=2, epochs=4000, eta=0.025, lam=0.03, seed=0):
    rng = np.random.default_rng(seed)
    n_users, n_items = R.shape
    P = 0.1*rng.normal(size=(n_users,k))
    Q = 0.1*rng.normal(size=(n_items,k))
    mu = np.nanmean(R)
    bu = np.zeros(n_users)
    bi = np.zeros(n_items)
    observed = np.argwhere(~np.isnan(R))
    losses=[]
    for epoch in range(epochs):
        rng.shuffle(observed)
        total=0.0
        for u,i in observed:
            pred = mu + bu[u] + bi[i] + P[u] @ Q[i]
            err = R[u,i] - pred
            total += err**2
            old_p = P[u].copy()
            bu[u] += eta*(err - lam*bu[u])
            bi[i] += eta*(err - lam*bi[i])
            P[u] += eta*(err*Q[i] - lam*P[u])
            Q[i] += eta*(err*old_p - lam*Q[i])
        if epoch % 50 == 0:
            reg = lam*(np.sum(P*P)+np.sum(Q*Q)+np.sum(bu*bu)+np.sum(bi*bi))
            losses.append(total/len(observed)+reg)
    return mu, bu, bi, P, Q, losses

mu, bu, bi, P, Q, losses = train_matrix_factorization(R, k=2, epochs=5000, eta=0.02, lam=0.04, seed=3)
R_hat = mu + bu[:,None] + bi[None,:] + P @ Q.T
pd.DataFrame(np.round(R_hat,2), index=users, columns=movies)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(len(losses))*50, losses)
plt.xlabel("Epoch")
plt.ylabel("Training objective")
plt.title("Matrix factorization training curve")
plt.tight_layout()
plt.show()

## 9. Recommend the top unseen item for each user

In [ ]:
recommendations=[]
for u, name in enumerate(users):
    unseen = np.where(np.isnan(R[u]))[0]
    if len(unseen)==0:
        continue
    best = unseen[np.argmax(R_hat[u, unseen])]
    recommendations.append((name, movies[best], R_hat[u,best]))

pd.DataFrame(recommendations, columns=["User", "Recommended item", "Predicted rating"])

## 10. Visualize hidden factors

The learned item vectors often reveal interpretable axes, though the axes can rotate or flip without changing predictions.

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(Q[:,0], Q[:,1], s=80)
for i, m in enumerate(movies):
    plt.text(Q[i,0]+0.01, Q[i,1]+0.01, m)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("Learned factor 1")
plt.ylabel("Learned factor 2")
plt.title("Items in learned latent factor space")
plt.tight_layout()
plt.show()

## 11. Evaluation with a train/test split

We hide some observed ratings, train on the rest, and test predictions on the hidden ratings.

In [ ]:
rng = np.random.default_rng(12)
obs = np.argwhere(~np.isnan(R))
rng.shuffle(obs)
test_size = max(3, len(obs)//5)
test = obs[:test_size]
train = obs[test_size:]

R_train = R.copy()
for u,i in test:
    R_train[u,i] = np.nan

mu2, bu2, bi2, P2, Q2, losses2 = train_matrix_factorization(R_train, k=2, epochs=5000, eta=0.02, lam=0.04, seed=8)
R_pred = mu2 + bu2[:,None] + bi2[None,:] + P2 @ Q2.T

errors=[]
for u,i in test:
    errors.append(R[u,i] - R_pred[u,i])
rmse = np.sqrt(np.mean(np.array(errors)**2))
rmse

## 12. High-dimensional ending: sparse recommendation matrices

Real recommendation matrices may have millions of users and items. The observed matrix is often extremely sparse, but the hidden factor dimension can be much smaller.

In [ ]:
n_users, n_items, k = 200, 120, 5
rng = np.random.default_rng(42)
P_true = rng.normal(size=(n_users,k))
Q_true = rng.normal(size=(n_items,k))
Full = P_true @ Q_true.T + 0.1*rng.normal(size=(n_users,n_items))

mask = rng.random(size=Full.shape) < 0.06
Sparse = np.where(mask, Full, np.nan)
observed_fraction = np.mean(~np.isnan(Sparse))
observed_fraction

In [ ]:
plt.figure(figsize=(8,4))
plt.imshow(~np.isnan(Sparse), aspect='auto')
plt.title(f"Sparse high-dimensional rating matrix: observed fraction = {observed_fraction:.3f}")
plt.xlabel("Items")
plt.ylabel("Users")
plt.tight_layout()
plt.show()

## Reflection questions

1. Why is missing not the same as zero?
2. When would user-based filtering work better than item-based filtering?
3. What does rank mean in a recommendation system?
4. How can recommendations create feedback loops?
5. How would you design a recommender for courses in an applied mathematics program?